In [11]:
pip install pandas numpy statsmodels scikit-learn scipy matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.


In [18]:
pip install prophet

Note: you may need to restart the kernel to use updated packages.


In [25]:
import numpy as np
import pandas as pd
import json
import time
import requests
from datetime import date
from prophet import Prophet
from statsmodels.tsa.statespace.sarimax import SARIMAX
from scipy.interpolate import PchipInterpolator

# Environmental Condition Dataset

In [34]:
# NASA POWER Daily API requires YYYYMMDD format
START, END = "20180101", "20260630"

STATES = {
    "Johor": ("Johor Bahru", 1.4655, 103.7578),
    "Kedah": ("Alor Setar", 6.1248, 100.3678),
    "Kelantan": ("Kota Bharu", 6.1399, 102.2422),
    "Melaka": ("Bandaraya Melaka", 2.1960, 102.2405),
    "Negeri Sembilan": ("Seremban", 2.7297, 101.9381),
    "Pahang": ("Kuantan", 3.7634, 103.2202),
    "Perak": ("Ipoh", 4.5841, 101.0829),
    "Perlis": ("Kangar", 6.4455, 100.2002),
    "Pulau Pinang": ("George Town", 5.4112, 100.3354),
    "Sabah": ("Kota Kinabalu", 5.9804, 116.0735),
    "Sarawak": ("Kuching", 1.5531, 110.3450),
    "Selangor": ("Shah Alam", 3.0733, 101.5185),
    "Terengganu": ("Kuala Terengganu", 5.3302, 103.1408),
    "WP Kuala Lumpur": ("Kuala Lumpur", 3.1409, 101.6932),
    "WP Putrajaya": ("Putrajaya", 2.9264, 101.6964),
    "WP Labuan": ("Labuan", 5.2852, 115.2478),
}

URL = "https://power.larc.nasa.gov/api/temporal/daily/point"
PARAMS = ["T2M", "RH2M", "PRECTOTCORR", "WS2M"]

frames = []
for state, (city, lat, lon) in STATES.items():
    r = requests.get(URL, params={
        "parameters": ",".join(PARAMS),
        "community": "AG",
        "latitude": lat, 
        "longitude": lon,
        "start": START, 
        "end": END,
        "format": "JSON",
    }, timeout=60)
    r.raise_for_status()
    payload = r.json()

    safe = state.replace(" ", "_")
    with open(f"raw_nasa_daily_{safe}.json", "w") as f:
        json.dump(payload, f)

    df = pd.DataFrame(payload["properties"]["parameter"])
    df.index.name = "ymd"
    df = df.reset_index()
    
    # Format YYYYMMDD into pandas datetime
    df["date"] = pd.to_datetime(df["ymd"], format="%Y%m%d")
    df = df.drop(columns=["ymd"])
    
    df = df.replace(-999, float("nan"))
    df.insert(0, "state", state)
    df.insert(1, "ibu_negeri", city)
    df.insert(2, "latitude", lat)
    df.insert(3, "longitude", lon)
    
    df = df.rename(columns={
        "T2M": "temperature_C",
        "RH2M": "humidity_pct",
        "PRECTOTCORR": "rainfall_mm_per_day",
        "WS2M": "wind_speed_m_s",
    })
    
    df = df[["state", "ibu_negeri", "latitude", "longitude", "date",
             "temperature_C", "humidity_pct", "rainfall_mm_per_day", "wind_speed_m_s"]]
    frames.append(df)
    print("OK:", state, len(df), "rows")
    time.sleep(1)

# Combine all state records
allc = pd.concat(frames, ignore_index=True)

# Strict date range filter from Jan 1, 2004 to June 30, 2026
allc = allc[(allc["date"] >= "2018-01-01") & (allc["date"] <= "2026-06-30")]

# Save full daily dataset across all states
allc.to_csv("weather_daily_all_states_2018_2026.csv", index=False)

# Save overall daily average across Malaysia
avg = (allc.groupby("date")[["temperature_C", "humidity_pct", "rainfall_mm_per_day", "wind_speed_m_s"]]
            .mean().reset_index())
avg.to_csv("weather_daily_malaysia_average_2018_2026.csv", index=False)

print(avg.head(10))
print("Total rows fetched per state:", len(allc) // len(STATES))
print("Missing values count:")
print(allc.isna().sum())
print("Downloaded at:", date.today())

OK: Johor 3103 rows
OK: Kedah 3103 rows
OK: Kelantan 3103 rows
OK: Melaka 3103 rows
OK: Negeri Sembilan 3103 rows
OK: Pahang 3103 rows
OK: Perak 3103 rows
OK: Perlis 3103 rows
OK: Pulau Pinang 3103 rows
OK: Sabah 3103 rows
OK: Sarawak 3103 rows
OK: Selangor 3103 rows
OK: Terengganu 3103 rows
OK: WP Kuala Lumpur 3103 rows
OK: WP Putrajaya 3103 rows
OK: WP Labuan 3103 rows
        date  temperature_C  humidity_pct  rainfall_mm_per_day  wind_speed_m_s
0 2018-01-01      25.407500     88.568125            25.757500        2.253750
1 2018-01-02      25.440000     89.220000            20.491875        2.121250
2 2018-01-03      25.816875     88.211875            17.138750        1.749375
3 2018-01-04      25.859375     88.568125            13.019375        1.528125
4 2018-01-05      25.646250     87.671250            11.860625        1.204375
5 2018-01-06      26.091875     85.533125             8.375625        1.311250
6 2018-01-07      26.079375     84.475000             5.902500        1.2

# Electricity Consumption Dataset

In [37]:
# 1. Load fail CSV asal
df = pd.read_csv('malaysia_monthly_electricity_consumption.csv')
df['date'] = pd.to_datetime(df['date'], format='%d/%m/%Y', errors='coerce')
df = df.sort_values('date').dropna(subset=['date', 'consumption_mkwh'])

monthly_series = df.set_index('date')['consumption_mkwh']

# 2. Latih model Prophet menggunakan data bulanan sebenar (2018 hingga Jun 2026)
prophet_df = pd.DataFrame({
    'ds': monthly_series.index,
    'y': monthly_series.values
})

model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,  # Mengambil kira turun naik hari bekerja vs hujung minggu
    daily_seasonality=False,
    seasonality_mode='multiplicative'
)
model.fit(prophet_df)

# 3. Tetapkan julat harian dari 1 Januari 2018 SEHINGGA 30 Jun 2026
daily_dates = pd.date_range(start=monthly_series.index.min(), end='2026-06-30', freq='D')
future_daily_df = pd.DataFrame({'ds': daily_dates})

# Ramal nilai harian kasar
forecast_daily = model.predict(future_daily_df)
daily_synthetic = forecast_daily[['ds', 'yhat']].rename(columns={'ds': 'date', 'yhat': 'daily_consumption'})
daily_synthetic['daily_consumption'] = daily_synthetic['daily_consumption'].clip(lower=0)

# 4. Skalakan nilai harian supaya jumlah sebulan tepat sama dengan data asal CSV awak
daily_synthetic['year_month'] = daily_synthetic['date'].dt.to_period('M')
monthly_sums = daily_synthetic.groupby('year_month')['daily_consumption'].transform('sum')

monthly_target_map = monthly_series.groupby(monthly_series.index.to_period('M')).first()
daily_synthetic['monthly_target'] = daily_synthetic['year_month'].map(monthly_target_map)

# Pengiraan akhir penjimatan/penskalaan harian
daily_synthetic['consumption_mkwh'] = (
    daily_synthetic['daily_consumption'] / monthly_sums
) * daily_synthetic['monthly_target']

# 5. Eksport fail akhir
final_daily_df = daily_synthetic[['date', 'consumption_mkwh']]
final_daily_df.to_csv('malaysia_daily_2018_to_2026_june30.csv', index=False)
print("Data harian dari Jan 2018 hingga 30 Jun 2026 berjaya dihasilkan!")

17:19:54 - cmdstanpy - INFO - Chain [1] start processing
17:19:54 - cmdstanpy - INFO - Chain [1] done processing


Data harian dari Jan 2018 hingga 30 Jun 2026 berjaya dihasilkan!


In [36]:
# 1. Load the original annual dataset
df = pd.read_csv('electricity_access_households.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# 2. Forecast recent anchor points to cover up to mid-2026
cagr_end = (
    df.loc[df['date'] == '2024-01-01', 'households'].values[0]
    / df.loc[df['date'] == '2021-01-01', 'households'].values[0]
) ** (1 / 3) - 1
val_2024 = df.loc[df['date'] == '2024-01-01', 'households'].values[0]

anchors = [
    {
        'date': pd.to_datetime('2025-01-01'),
        'households': val_2024 * ((1 + cagr_end) ** 1),
    },
    {
        'date': pd.to_datetime('2026-07-01'),
        'households': val_2024 * ((1 + cagr_end) ** 2.5),
    },
]

df_anchors = pd.concat(
    [pd.DataFrame(anchors), df[['date', 'households']]]
).sort_values('date').reset_index(drop=True)

# 3. Target Date Range: January 1, 2018 to June 1, 2026
target_dates = pd.date_range(start='2018-01-01', end='2026-06-30', freq='D')
x_anchors = df_anchors['date'].map(pd.Timestamp.toordinal).values
y_anchors = df_anchors['households'].values

# 4. Apply PCHIP Interpolation
pchip = PchipInterpolator(x_anchors, y_anchors)
x_target = target_dates.map(pd.Timestamp.toordinal).values
daily_households = pchip(x_target)

# 5. Integrate Weather Modulation & Round to Whole Numbers
np.random.seed(42)
days_of_year = target_dates.dayofyear
weather_modulation = (
    1.0
    + 0.002 * np.sin(2 * np.pi * days_of_year / 365)
    + 0.0005 * np.random.randn(len(target_dates))
)

# Build final DataFrame
synthetic_daily_df = pd.DataFrame({
    'date': target_dates,
    'state': 'Malaysia',
    'base_households': np.round(daily_households).astype(int),
    'weather_modulation': weather_modulation,
    'synthetic_households': np.round(daily_households * weather_modulation).astype(
        int
    ),
})

# Save to CSV
synthetic_daily_df.to_csv(
    'malaysia_daily_households_2018_2026.csv', index=False
)
print(synthetic_daily_df.head())

        date     state  base_households  weather_modulation  \
0 2018-01-01  Malaysia          8452376            1.000283   
1 2018-01-02  Malaysia          8452961            1.000000   
2 2018-01-03  Malaysia          8453545            1.000427   
3 2018-01-04  Malaysia          8454130            1.000899   
4 2018-01-05  Malaysia          8454714            1.000055   

   synthetic_households  
0               8454766  
1               8452958  
2               8457156  
3               8461731  
4               8455178  
